# Tomahawk PF/CS FEA + native-solver Kaggle run

This notebook is intentionally a two-file Kaggle workflow: upload this notebook and `Tomahawk_Kaggle_Data.zip`. It uses the current PF/CS engineering closure, performs a real NumPy 3-D truss finite-element screen, and then attempts native-solver evidence only when a real solver package and MAKEGRID/mgrid file are present.

No Internet, `pip install`, synthetic equilibrium, or fabricated external-laboratory result is allowed. A missing solver is reported as an input condition. A completed FEA screen is not engineering qualification.

In [1]:
from pathlib import Path
import csv, hashlib, importlib, json, os, shutil, subprocess, sys, time, zipfile
import numpy as np

print('KAGGLE_BOOTSTRAP_START')
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd() / '_kaggle_work'
WORK_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working'), Path('/kaggle/models'), Path.cwd()]
SEARCH_ROOTS = [p for p in SEARCH_ROOTS if p.exists()]

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f'unsafe ZIP member: {member.filename}')
        z.extractall(destination)

def locate_data_root():
    archives = sorted({p.resolve() for root in SEARCH_ROOTS for p in root.rglob('Tomahawk_Kaggle_Data.zip') if p.is_file()})
    if len(archives) > 1:
        raise RuntimeError('INPUT_BLOCKED: more than one Tomahawk_Kaggle_Data.zip found')
    if archives:
        extracted = WORK_DIR / 'tomahawk_kaggle_extracted'
        if extracted.exists():
            shutil.rmtree(extracted)
        extracted.mkdir(parents=True)
        safe_extract(archives[0], extracted)
        roots = sorted(p for p in extracted.rglob('kaggle_data_manifest.json') if p.is_file())
        if len(roots) != 1:
            raise RuntimeError('INPUT_BLOCKED: extracted data manifest is missing or ambiguous')
        return roots[0].parent
    roots = sorted({p.parent.resolve() for root in SEARCH_ROOTS for p in root.rglob('kaggle_data_manifest.json') if (p.parent / 'MHD/mhd_input.json').is_file()})
    if len(roots) != 1:
        raise RuntimeError('INPUT_BLOCKED: unpacked data root is missing or ambiguous')
    return roots[0]

try:
    DATA_ROOT = locate_data_root()
    data_manifest = json.loads((DATA_ROOT / 'kaggle_data_manifest.json').read_text())
    mismatches = []
    for record in data_manifest['files']:
        path = (DATA_ROOT / record['path']).resolve()
        if DATA_ROOT.resolve() not in path.parents or not path.is_file():
            mismatches.append(record['path'])
            continue
        if path.stat().st_size != record['size_bytes'] or sha256(path) != record['sha256']:
            mismatches.append(record['path'])
    required = ['MHD/mhd_input.json', 'MHD/mhd_profile_sweep.csv', 'MHD/cad_a2_coil_centerlines.csv', 'MHD/cad_a2_coil_currents.csv', 'CFD/cfd_case_manifest.json', 'CFD/cfd_cht_parameter_sweep.csv', 'CHF/chf_input.json', 'NEUTRONICS/material_region_map.csv', 'NEUTRONICS/openmc_input_manifest.json', 'NEUTRONICS/source_sweep.csv', 'MHD/engineering/cs_module_table.csv', 'MHD/engineering/cs_precompression_screen.json']
    missing = [name for name in required if not (DATA_ROOT / name).is_file()]
    if mismatches or missing:
        raise RuntimeError(json.dumps({'status': 'INPUT_BLOCKED', 'hash_mismatches': mismatches, 'missing': missing}))
    print('ONE_CLICK_INPUT_VERIFIED')
    print('data_root =', DATA_ROOT)
except Exception as exc:
    print(str(exc))
    raise

KAGGLE_BOOTSTRAP_START
ONE_CLICK_INPUT_VERIFIED
data_root = /kaggle/input/datasets/nicolebryant29/tomahawk-kaggle-data1/Tomahawk_Kaggle_Data


In [2]:
# Real NumPy 3-D truss FEA screen for the six-module CS stack.
# This is a transparent support/load model, not a released casing or coil FEA.

def read_csv_rows(path):
    with Path(path).open(newline='') as handle:
        return list(csv.DictReader(handle))

module_rows = read_csv_rows(DATA_ROOT / 'MHD/engineering/cs_module_table.csv')
preload_doc = json.loads((DATA_ROOT / 'MHD/engineering/cs_precompression_screen.json').read_text())
scenario_name = 'all_co_directional_conservative'
module_forces = np.asarray(preload_doc['scenarios'][scenario_name]['module_generalized_axial_force_N'], dtype=float)
E = 150.0e9
support_area = 0.10
stack_area = 0.10
allowable_stress = 250.0e6
support_offset = 0.80
module_count = len(module_rows)
nodes = []
coil_nodes = []
anchor_nodes = []
for row in module_rows:
    i = int(row['module_index'])
    z = (i - 0.5 * (module_count - 1)) * float(row['module_height_m'])
    r = 0.5 * (float(row['inner_winding_radius_m']) + float(row['outer_winding_radius_m']))
    coil_nodes.append(len(nodes)); nodes.append([r, 0.0, z])
    anchors = []
    for theta in (0.0, 2.0 * np.pi / 3.0, 4.0 * np.pi / 3.0):
        anchors.append(len(nodes))
        nodes.append([r + support_offset * np.cos(theta), support_offset * np.sin(theta), z + 0.40 * np.sin(theta)])
    anchor_nodes.extend(anchors)
nodes = np.asarray(nodes, dtype=float)
elements = []
for i, coil in enumerate(coil_nodes):
    for anchor in anchor_nodes[3*i:3*i+3]:
        elements.append((coil, anchor, support_area, 'support'))
for i in range(module_count - 1):
    elements.append((coil_nodes[i], coil_nodes[i+1], stack_area, 'stack'))
K = np.zeros((3 * len(nodes), 3 * len(nodes)))
def add_truss_stiffness(matrix, a, b, area):
    delta = nodes[b] - nodes[a]
    length = float(np.linalg.norm(delta))
    direction = delta / length
    block = E * area / length * np.outer(direction, direction)
    dofs_a = slice(3*a, 3*a+3); dofs_b = slice(3*b, 3*b+3)
    matrix[dofs_a, dofs_a] += block; matrix[dofs_b, dofs_b] += block
    matrix[dofs_a, dofs_b] -= block; matrix[dofs_b, dofs_a] -= block
for a, b, area, kind in elements:
    add_truss_stiffness(K, a, b, area)
F = np.zeros(3 * len(nodes))
for i, node in enumerate(coil_nodes):
    F[3*node + 2] = module_forces[i]
fixed = sorted(dof for node in anchor_nodes for dof in (3*node, 3*node+1, 3*node+2))
free = np.asarray([dof for dof in range(3 * len(nodes)) if dof not in set(fixed)], dtype=int)
u = np.zeros(3 * len(nodes))
u[free] = np.linalg.solve(K[np.ix_(free, free)], F[free])
element_results = []
for a, b, area, kind in elements:
    delta = nodes[b] - nodes[a]; length = float(np.linalg.norm(delta)); direction = delta / length
    extension = float(np.dot(u[3*b:3*b+3] - u[3*a:3*a+3], direction))
    axial_force = E * area / length * extension
    element_results.append({'node_a': a, 'node_b': b, 'kind': kind, 'length_m': length, 'area_m2': area, 'axial_force_N': axial_force, 'stress_Pa': axial_force / area})
max_stress = max(abs(item['stress_Pa']) for item in element_results)
fea_screen = {
    'schema_version': 'TOMAHAWK-KAGGLE-3D-TRUSS-FEA-SCREEN-0.1',
    'status': 'SCREENING_RESULT_NOT_QUALIFIED',
    'model': {'element_type': '3-D axial truss', 'nodes': int(len(nodes)), 'elements': int(len(elements)), 'E_Pa': E, 'support_area_m2': support_area, 'stack_area_m2': stack_area, 'allowable_stress_Pa_assumption': allowable_stress, 'load_case': scenario_name},
    'results': {'max_displacement_m': float(np.max(np.abs(u))), 'max_element_stress_Pa': float(max_stress), 'max_element_stress_MPa': float(max_stress / 1.0e6), 'required_area_at_allowable_stress_m2': float(max(abs(item['axial_force_N']) for item in element_results) / allowable_stress), 'element_results': element_results},
    'acceptance': {'linear_system_solved': True, 'stress_screen_pass': bool(max_stress <= allowable_stress), 'engineering_qualification': False},
    'limitations': ['Support geometry and material properties are explicit screening assumptions, not released hardware.', 'No contact, preload, cooldown, Lorentz body-force map, casing, insulation, fatigue, fracture, or nonlinear material behavior is solved.', 'The result cannot replace a 3-D electromagnetic/structural FEA package.']
}
print('SMOKE_PASS')
print('FEA stress screen:', 'PASS' if fea_screen['acceptance']['stress_screen_pass'] else 'OPEN_REVISE_SUPPORT_MODEL', '| max stress MPa =', fea_screen['results']['max_element_stress_MPa'])

SMOKE_PASS
FEA stress screen: OPEN_REVISE_SUPPORT_MODEL | max stress MPa = 11036.914483493416


In [3]:
# Native-solver adapter. It executes only an explicitly packaged, hashed solver.
solver_manifest = json.loads((DATA_ROOT / 'solver_package_manifest.json').read_text())
solver_type = solver_manifest.get('package_type', 'none')
native = {'schema_version': 'TOMAHAWK-KAGGLE-NATIVE-SOLVER-EVIDENCE-0.1', 'package_type': solver_type, 'status': None, 'mgrid': {}, 'execution': {}, 'qualification': {'free_boundary_3D_MHD_pass': False}}
mgrid_path = solver_manifest.get('mgrid_path')
if mgrid_path:
    mgrid = (DATA_ROOT / mgrid_path).resolve()
    if mgrid.is_file() and DATA_ROOT.resolve() in mgrid.parents:
        actual_mgrid_hash = sha256(mgrid)
        expected_mgrid_hash = solver_manifest.get('mgrid_sha256')
        native['mgrid'] = {'path': str(mgrid), 'sha256': actual_mgrid_hash, 'present': not expected_mgrid_hash or actual_mgrid_hash == expected_mgrid_hash, 'hash_match': not expected_mgrid_hash or actual_mgrid_hash == expected_mgrid_hash}
    else:
        native['mgrid'] = {'path': mgrid_path, 'present': False}
else:
    native['mgrid'] = {'present': False, 'reason': 'no mgrid_path declared'}

def run_hashed_binary():
    entry = (DATA_ROOT / solver_manifest['entrypoint']).resolve()
    if DATA_ROOT.resolve() not in entry.parents or not entry.is_file():
        return {'status': 'INPUT_BLOCKED', 'reason': 'binary entrypoint missing'}
    expected = solver_manifest.get('sha256')
    if not expected or sha256(entry) != expected:
        return {'status': 'INPUT_BLOCKED', 'reason': 'binary hash missing or mismatched'}
    args = solver_manifest.get('entrypoint_args', [])
    if not isinstance(args, list) or not all(isinstance(x, str) for x in args):
        return {'status': 'INPUT_BLOCKED', 'reason': 'entrypoint_args must be a string list'}
    try:
        result = subprocess.run([str(entry), *args], cwd=str(entry.parent), capture_output=True, text=True, timeout=3600)
    except Exception as exc:
        return {'status': 'EXECUTION_ERROR', 'reason': str(exc)}
    (WORK_DIR / 'native_solver_stdout.txt').write_text(result.stdout)
    (WORK_DIR / 'native_solver_stderr.txt').write_text(result.stderr)
    return {'status': 'NATIVE_SOLVER_EXECUTED' if result.returncode == 0 else 'NATIVE_SOLVER_FAILED', 'returncode': result.returncode, 'stdout_path': str(WORK_DIR / 'native_solver_stdout.txt'), 'stderr_path': str(WORK_DIR / 'native_solver_stderr.txt')}

if solver_type == 'binary':
    native['execution'] = run_hashed_binary()
elif solver_type == 'python_module':
    module_root = (DATA_ROOT / solver_manifest.get('module_root', '')).resolve()
    if module_root.is_dir() and DATA_ROOT.resolve() in module_root.parents:
        sys.path.insert(0, str(module_root))
        try:
            importlib.import_module(solver_manifest['module'])
            native['execution'] = {'status': 'MODULE_IMPORT_PASS', 'module': solver_manifest['module'], 'note': 'A solver callable and declared outputs are still required for execution evidence.'}
        except Exception as exc:
            native['execution'] = {'status': 'NATIVE_SOLVER_FAILED', 'reason': repr(exc)}
    else:
        native['execution'] = {'status': 'INPUT_BLOCKED', 'reason': 'packaged module_root missing'}
elif solver_type == 'source_archive':
    native['execution'] = {'status': 'INPUT_BLOCKED_SOURCE_BUILD_REQUIRED', 'reason': 'source archives are not compiled or installed automatically; provide a reproducible local build contract and generated executable'}
else:
    native['execution'] = {'status': 'INPUT_BLOCKED', 'reason': 'no native solver package declared'}
native['status'] = native['execution']['status']
native['qualification']['mgrid_present'] = bool(native['mgrid'].get('present'))
native['qualification']['native_solver_executed'] = native['status'] == 'NATIVE_SOLVER_EXECUTED'
native['qualification']['free_boundary_3D_MHD_pass'] = bool(native['qualification']['mgrid_present'] and native['qualification']['native_solver_executed'])
print('NATIVE_SOLVER_STATUS:', native['status'])
print('native solver pass:', native['qualification']['free_boundary_3D_MHD_pass'])

NATIVE_SOLVER_STATUS: INPUT_BLOCKED
native solver pass: False


In [4]:
# Write a self-contained evidence package for Claude or another reviewer.
results_dir = WORK_DIR / 'Tomahawk_Kaggle_Results'
if results_dir.exists():
    shutil.rmtree(results_dir)
results_dir.mkdir(parents=True)
def write_json(name, doc):
    (results_dir / name).write_text(json.dumps(doc, indent=2) + '\n')
write_json('fea_screen_results.json', fea_screen)
write_json('native_solver_evidence.json', native)
write_json('run_summary.json', {
    'schema_version': 'TOMAHAWK-KAGGLE-FEA-NATIVE-RUN-0.1',
    'status': 'FEA_SCREEN_COMPLETE_NATIVE_SOLVER_OPEN' if not native['qualification']['free_boundary_3D_MHD_pass'] else 'FEA_AND_NATIVE_SOLVER_EVIDENCE_COMPLETE_QUALIFICATION_OPEN',
    'data_root': str(DATA_ROOT),
    'fea_stress_screen_pass': fea_screen['acceptance']['stress_screen_pass'],
    'native_solver_status': native['status'],
    'free_boundary_3D_MHD_pass': False,
    'qualified_design_pass': False,
    'external_execution_claim': False,
    'next_required_inputs': data_manifest['required_not_synthesized']
})
with zipfile.ZipFile(WORK_DIR / 'Tomahawk_Kaggle_Results.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(results_dir.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(results_dir).as_posix())
print('FULL_RUN_COMPLETE')
print('FULL_SCREEN_PASS' if fea_screen['acceptance']['stress_screen_pass'] else 'SCREENING_FAIL_REVISE')
print('SCREENING_PASS_WITH_EXTERNAL_BLOCKERS' if fea_screen['acceptance']['stress_screen_pass'] and not native['qualification']['free_boundary_3D_MHD_pass'] else 'NATIVE_SOLVER_EVIDENCE_COMPLETE_OR_FEA_REVISE')
print('RESULT_ZIP:', WORK_DIR / 'Tomahawk_Kaggle_Results.zip')

FULL_RUN_COMPLETE
SCREENING_FAIL_REVISE
NATIVE_SOLVER_EVIDENCE_COMPLETE_OR_FEA_REVISE
RESULT_ZIP: /kaggle/working/Tomahawk_Kaggle_Results.zip
